# 2f-deep — Koszul Jacobi $\Leftrightarrow [\pi,\pi]_{SN}=0$ (engine, sıfırdan)

**Problem (f).** $T^*M$ Koszul bracket'inin Jacobi özdeşliği:

$$
\sum_{\text{cyc}}\bigl[\alpha,[\beta,\gamma]_K\bigr]_K = 0
\qquad \Longleftrightarrow \qquad [\pi,\pi]_{SN} = 0.
$$

**2f-theo** bu özdeşliği `theorem_book`'tan tek-adım olarak cite eder. **2f-deep** aynı sonucu sıfırdan ispatlar — Faz 13'ün altı yeni rewrite axiom'unu yükler, cyclic LHS'i 27 ground term'a açar, ve operatör-komütatör + Lie-Jacobi-VF + SN-bivector formülü zinciriyle inert `[·,·]_SN(π, π)` node'una iner.

**Faz 13 axiom paketi.**

| # | Axiom | Modül | Rol |
|---|---|---|---|
| 1 | $\pi^\sharp(A+B+\dots) \to \pi^\sharp(A)+\pi^\sharp(B)+\dots$ | `sharp_axioms.SharpLinearityDefinition` | Sharp R-lineerliği |
| 2 | $\pi^\sharp(df) \to X_f$ | `sharp_axioms.SharpOnExactDefinition` | Hamiltonian VF tanımı |
| 3 | $\langle A+B, X\rangle \to \langle A,X\rangle+\langle B,X\rangle$ (Neg slotunu da kapsar) | `pairing_axioms.PairingLinearityDefinition` | Pairing R-lineerliği |
| 4 | $L_X\langle\alpha, Y\rangle \to \langle L_X\alpha, Y\rangle + \langle\alpha, L_X Y\rangle$ | `pairing_axioms.PairingLieLeibnizDefinition` | Pairing-Lie Leibniz |
| 5a | $L_X\circ L_Y - L_Y\circ L_X \to L_{[X,Y]_{VF}}$ | `vf_axioms.OpCommutatorVfDefinition` | Operatör komütatörü |
| 5b | Cyclic $[X,[Y,Z]_{VF}]_{VF}$ üçlüsü $\to 0$ | `vf_axioms.LieVfJacobiDefinition` | Lie-Jacobi for VF |
| 6 | Cyclic $L_{[\pi^\sharp a, \pi^\sharp b]_{VF}}(c) \to [\cdot,\cdot]_{SN}(\pi, \pi)$ | `sn_axiom.SnBivectorFormulaDefinition` | SN bivector formülü |

Engine ek olarak iki *bookkeeping helper* kullanır (notebook-local): `KoszulExpandDefinition` (her bir `BracketApply([·,·]_K, ·, ·)` node'unu Cartan formülüne açar) ve `LieRLinearityInVfDefinition` ($L_{X+Y}=L_X+L_Y$ — `LieDerivative` atom olduğu için engine onun `vector_field` slotuna doğal olarak inmez; bu kural distribution'ı dış katmana taşır).

Bu defter, engine'in zincirini adım adım izler ve Axiom 6'nın kapanış noktasını gösterir.

In [1]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'jacopy' / '__init__.py').is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.algebra.derivation import Act
from jacopy.brackets.base import BracketApply
from jacopy.brackets.koszul import KoszulBracket
from jacopy.calculus.lie_derivative import LieDerivative, lie_derivative
from jacopy.calculus.musical import sharp
from jacopy.calculus.sharp_axioms import SharpLinearityDefinition, SharpOnExactDefinition
from jacopy.calculus.pairing_axioms import PairingLinearityDefinition, PairingLieLeibnizDefinition
from jacopy.calculus.vf_axioms import OpCommutatorVfDefinition, LieVfJacobiDefinition
from jacopy.calculus.sn_axiom import SnBivectorFormulaDefinition
from jacopy.calculus.linearity_axioms import (
    LieDerivativeArgLinearityDefinition,
    ExteriorDerivativeLinearityDefinition,
)
from jacopy.core.expr import Symbol, Sum, Neg
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import (
    Definition, ExpansionEngine,
    DSquaredZeroDefinition, LieDerivativeCommutesWithDDefinition,
)
from jacopy.algorithms.product_rule import product_rule
from jacopy.algorithms.simplify import simplify

## 1. Kurulum — bivector, generic 1-formlar, KoszulBracket

$\pi$ bivector, $\alpha, \beta, \gamma$ generic 1-formlar, anchor $= \pi^\sharp$. Engine flow-mode `L_X` kullanır; cartan-mode magic-formula açılımı bu seviyede gereksiz patlamaya yol açar.

In [2]:
reg = PropertyRegistry()
pi = Symbol('π'); reg.declare(pi, Graded(degree=1))
alpha = Symbol('α'); reg.declare(alpha, Graded(degree=1))
beta = Symbol('β');  reg.declare(beta,  Graded(degree=1))
gamma = Symbol('γ'); reg.declare(gamma, Graded(degree=1))

pi_sharp = sharp(pi)
flow_lie = lambda X: lie_derivative(X, definition='flow')
K = KoszulBracket(anchor=pi_sharp, lie_derivative=flow_lie)
print(f'π            : {pi}  (bivector)')
print(f'α, β, γ      : {alpha}, {beta}, {gamma}')
print(f'KoszulBracket: {K.name}, anchor={pi_sharp}')

π            : π  (bivector)
α, β, γ      : α, β, γ
KoszulBracket: [·,·]_K, anchor=π♯


## 2. LHS — cyclic Koszul Jacobi sum

`graded_jacobi_obstruction(α, β, γ)` 3 inert `BracketApply([·,·]_K, ·, ·)` node'undan oluşan Sum üretir. Koszul $\deg=0$ + $|α|=1$ olduğundan parity $1$'e düşer ve her terim $\mathrm{Neg}$ ile sarılır.

In [3]:
lhs = K.graded_jacobi_obstruction(alpha, beta, gamma, registry=reg)
print('LHS structure:')
for i, c in enumerate(lhs.children):
    print(f'  [{i}] {c}')

LHS structure:
  [0] (-[·,·]_K(α, [·,·]_K(β, γ)))
  [1] (-[·,·]_K(β, [·,·]_K(γ, α)))
  [2] (-[·,·]_K(γ, [·,·]_K(α, β)))


## 3. Yardımcı kurallar — Koszul açılımı + L_X-in-X dağılımı

Defter üç engine pipeline'ı çalıştırarak LHS'i giderek daha kapalı bir forma indirir. Ortak yardımcılar:

* `KoszulExpandDefinition` — `BracketApply([·,·]_K, a, b)` görüldüğünde Koszul tanımını açar (her bracket → 3 alt-terim).
* `LieRLinearityInVfDefinition` — `L_X` operatörünün vektör alanı slot'unda Sum/Neg dağıtır (engine bu slot'a giremez, çünkü `LieDerivative` atomdur).

Diğer dağıtım kuralları paket içinden gelir: `LieDerivativeArgLinearityDefinition`, `ExteriorDerivativeLinearityDefinition`, `SharpLinearityDefinition`, `PairingLinearityDefinition`, `PairingLieLeibnizDefinition`.

In [4]:
class KoszulExpandDefinition(Definition):
    name = 'Koszul expand'
    def __init__(self, bracket): self._b = bracket
    def matches(self, e):
        return isinstance(e, BracketApply) and e.bracket == self._b
    def rewrite(self, e):
        return self._b.expand(e.a, e.b)


class LieRLinearityInVfDefinition(Definition):
    name = 'L_X R-linearity in X'
    def matches(self, e):
        if not (isinstance(e, Act) and isinstance(e.op, LieDerivative)):
            return False
        v = e.op.vector_field
        if isinstance(v, (Sum, Neg)):
            return True
        return isinstance(v, Act) and isinstance(v.arg, (Sum, Neg))
    def rewrite(self, e):
        L = e.op; v = L.vector_field; arg = e.arg
        defn = L.definition
        if isinstance(v, Neg):
            return Neg(Act(lie_derivative(v.arg, definition=defn), arg))
        if isinstance(v, Sum):
            terms = []
            for c in v.children:
                if isinstance(c, Neg):
                    terms.append(Neg(Act(lie_derivative(c.arg, definition=defn), arg)))
                else:
                    terms.append(Act(lie_derivative(c, definition=defn), arg))
            return Sum.make(*terms)
        # Act(Op, Sum/Neg) — tek katman daha derin, distribution'ı dış katmana taşı.
        op = v.op; inner = v.arg
        if isinstance(inner, Neg):
            return Neg(Act(lie_derivative(Act(op, inner.arg), definition=defn), arg))
        terms = []
        for c in inner.children:
            if isinstance(c, Neg):
                terms.append(Neg(Act(lie_derivative(Act(op, c.arg), definition=defn), arg)))
            else:
                terms.append(Act(lie_derivative(Act(op, c), definition=defn), arg))
        return Sum.make(*terms)

## 4. Stage 1 — yalnızca *açılım* kuralları → 27 ham terim

İlk pipeline yalnızca tanım-açıcı kuralları içerir; HİÇBİR folding/recognizer kuralı yok. Çıktı: cyclic Koszul Jacobi sum'ın tam ham 27-terimlik flat hâli.

In [5]:
def run_engine(definitions, lhs_in, max_iter=80):
    """Engine fix-point + product_rule + simplify cascade."""
    engine = ExpansionEngine(definitions)
    current = lhs_in
    all_steps = []
    for it in range(max_iter):
        expanded, steps = engine.expand(current)
        all_steps.extend(steps)
        after = simplify(product_rule(expanded, reg), reg)
        if after == current:
            break
        current = after
    return current, all_steps, it


# --- Stage 1 — *only* expansion / linearity rules (no folding) ---
expansion_only = [
    KoszulExpandDefinition(K),                          # bookkeeping
    SharpLinearityDefinition(pi_sharp),                 # 13.A axiom 1
    PairingLinearityDefinition(),                       # 13.B axiom 3
    PairingLieLeibnizDefinition(),                      # 13.B axiom 4
    LieRLinearityInVfDefinition(),                      # bookkeeping
    LieDerivativeArgLinearityDefinition(),              # supplementary
    ExteriorDerivativeLinearityDefinition(),            # supplementary
]
stage1, steps1, it1 = run_engine(expansion_only, lhs)
print(f"Stage 1 fix-point: {it1} iter, {len(steps1)} expand-step")
from collections import Counter
for rule, n in Counter(s.rule for s in steps1).most_common():
    print(f"  {n:3d}× {rule}")

print()
if isinstance(stage1, Sum):
    print(f"Toplam ham terim: {len(stage1.children)}")
    print()
    for i, c in enumerate(stage1.children):
        print(f"  [{i:2d}] {c}")
else:
    print(f"(Sum değil) {stage1}")

Stage 1 fix-point: 1 iter, 24 expand-step
    9× Pairing R-linearity
    6× Koszul expand
    3× L_X R-linearity in arg
    3× L_X R-linearity in X
    3× d R-linearity

Toplam ham terim: 27

  [ 0] L_π♯(L_π♯(α)(β))(γ)
  [ 1] (-L_π♯(L_π♯(α)(γ))(β))
  [ 2] (-L_π♯(L_π♯(β)(α))(γ))
  [ 3] L_π♯(L_π♯(β)(γ))(α)
  [ 4] L_π♯(L_π♯(γ)(α))(β)
  [ 5] (-L_π♯(L_π♯(γ)(β))(α))
  [ 6] (-L_π♯(d(⟨π♯(α), β⟩))(γ))
  [ 7] (-L_π♯(d(⟨π♯(β), γ⟩))(α))
  [ 8] (-L_π♯(d(⟨π♯(γ), α⟩))(β))
  [ 9] (-L_π♯(α)(L_π♯(β)(γ)))
  [10] L_π♯(α)(L_π♯(γ)(β))
  [11] L_π♯(α)(d(⟨π♯(β), γ⟩))
  [12] L_π♯(β)(L_π♯(α)(γ))
  [13] (-L_π♯(β)(L_π♯(γ)(α)))
  [14] L_π♯(β)(d(⟨π♯(γ), α⟩))
  [15] (-L_π♯(γ)(L_π♯(α)(β)))
  [16] L_π♯(γ)(L_π♯(β)(α))
  [17] L_π♯(γ)(d(⟨π♯(α), β⟩))
  [18] d(⟨π♯(α), L_π♯(β)(γ)⟩)
  [19] (-d(⟨π♯(α), L_π♯(γ)(β)⟩))
  [20] (-d(⟨π♯(α), d(⟨π♯(β), γ⟩)⟩))
  [21] (-d(⟨π♯(β), L_π♯(α)(γ)⟩))
  [22] d(⟨π♯(β), L_π♯(γ)(α)⟩)
  [23] (-d(⟨π♯(β), d(⟨π♯(γ), α⟩)⟩))
  [24] d(⟨π♯(γ), L_π♯(α)(β)⟩)
  [25] (-d(⟨π♯(γ), L_π♯(β)(α)⟩))
  [26] (-d(⟨π♯(γ

## 5. Stage 2 — folding ekle (`OpCommutator` + `SN bivector formula`)

Şimdi 13.C ve 13.D recognizer axiom'larını engine'a ekliyoruz:

* **`OpCommutatorVfDefinition`** — `[L_X, L_Y] = L_{[X,Y]_VF}` (13.C-5).
* **`LieVfJacobiDefinition`** — `L_{[X,Y]_VF}` üçlü cyclic kapanışı (13.C-5b).
* **`SnBivectorFormulaDefinition`** — `Σ_cyc L_{[π♯a, π♯b]_VF}(c) → ½⟨[π,π]_SN, α∧β∧γ⟩` (13.D-6).

Beklenti: 27 ham terimden 6 tanesi (gerçek `L_X(L_Y(c))` shape'leri) folding ile **bir adet** `BracketApply([·,·]_SN, π, π)` token'ına katlanır. Geriye kalan ~22 terim — *farklı* evaluator parçalarıdır (nested-vector-field L'ler + Hamiltonian-VF L'ler + d-pairing'ler).

In [6]:
folding_defs = expansion_only + [
    OpCommutatorVfDefinition(lie_derivative_factory=flow_lie),  # 13.C axiom 5a
    LieVfJacobiDefinition(),                                    # 13.C axiom 5b
    SnBivectorFormulaDefinition(pi_sharp),                      # 13.D axiom 6
]
stage2, steps2, it2 = run_engine(folding_defs, lhs)
print(f"Stage 2 fix-point: {it2} iter, {len(steps2)} expand-step")
for rule, n in Counter(s.rule for s in steps2).most_common():
    print(f"  {n:3d}× {rule}")

print()
if isinstance(stage2, Sum):
    sn = sum(1 for c in stage2.children if isinstance(c, BracketApply))
    print(f"Toplam terim: {len(stage2.children)}  ({sn} adet SN handle, {len(stage2.children)-sn} residue)")
    print()
    for i, c in enumerate(stage2.children):
        marker = "  ★" if isinstance(c, BracketApply) else "   "
        print(f"  [{i:2d}]{marker} {c}")
else:
    print(f"(Sum değil) {stage2}")

Stage 2 fix-point: 2 iter, 28 expand-step
    9× Pairing R-linearity
    6× Koszul expand
    3× L_X R-linearity in arg
    3× L_X R-linearity in X
    3× d R-linearity
    3× [L_X, L_Y] = L_{[X,Y]_VF}
    1× [π,π]_SN bivector formula

Toplam terim: 22  (1 adet SN handle, 21 residue)

  [ 0]    L_π♯(L_π♯(α)(β))(γ)
  [ 1]    (-L_π♯(L_π♯(α)(γ))(β))
  [ 2]    (-L_π♯(L_π♯(β)(α))(γ))
  [ 3]    L_π♯(L_π♯(β)(γ))(α)
  [ 4]    L_π♯(L_π♯(γ)(α))(β)
  [ 5]    (-L_π♯(L_π♯(γ)(β))(α))
  [ 6]    (-L_π♯(d(⟨π♯(α), β⟩))(γ))
  [ 7]    (-L_π♯(d(⟨π♯(β), γ⟩))(α))
  [ 8]    (-L_π♯(d(⟨π♯(γ), α⟩))(β))
  [ 9]    L_π♯(α)(d(⟨π♯(β), γ⟩))
  [10]    L_π♯(β)(d(⟨π♯(γ), α⟩))
  [11]    L_π♯(γ)(d(⟨π♯(α), β⟩))
  [12]  ★ [·,·]_SN(π, π)
  [13]    d(⟨π♯(α), L_π♯(β)(γ)⟩)
  [14]    (-d(⟨π♯(α), L_π♯(γ)(β)⟩))
  [15]    (-d(⟨π♯(α), d(⟨π♯(β), γ⟩)⟩))
  [16]    (-d(⟨π♯(β), L_π♯(α)(γ)⟩))
  [17]    d(⟨π♯(β), L_π♯(γ)(α)⟩)
  [18]    (-d(⟨π♯(β), d(⟨π♯(γ), α⟩)⟩))
  [19]    d(⟨π♯(γ), L_π♯(α)(β)⟩)
  [20]    (-d(⟨π♯(γ), L_π♯(β)(α)⟩))
  [21]  

## 6. Stage 3 — `[L_X, d] = 0` + `d² = 0` ekle → residue daha da iner

Klasik Cartan kimlikleri:

* **`LieDerivativeCommutesWithDDefinition`** — `L_X(d ω) → d(L_X ω)` (flow-mode için engine-level).
* **`DSquaredZeroDefinition`** — `d(d(x)) → 0`.

Bunlar residue'daki `L_π♯a(d⟨…⟩)` blokların bir kısmını `d⟨L_π♯a(π♯b), c⟩ + d⟨π♯b, L_π♯a c⟩` yapısına dönüştürür; bazı pairing pair'i kendi içinde simplify ile sıfırlanır.

In [7]:
pushdown_defs = folding_defs + [
    LieDerivativeCommutesWithDDefinition(),   # [L_X, d] = 0
    DSquaredZeroDefinition(),                  # d² = 0
]
stage3, steps3, it3 = run_engine(pushdown_defs, lhs)
print(f"Stage 3 fix-point: {it3} iter, {len(steps3)} expand-step")
for rule, n in Counter(s.rule for s in steps3).most_common():
    print(f"  {n:3d}× {rule}")

print()
if isinstance(stage3, Sum):
    sn = sum(1 for c in stage3.children if isinstance(c, BracketApply))
    print(f"Toplam terim: {len(stage3.children)}  ({sn} adet SN handle, {len(stage3.children)-sn} residue)")
    print()
    for i, c in enumerate(stage3.children):
        marker = "  ★" if isinstance(c, BracketApply) else "   "
        print(f"  [{i:2d}]{marker} {c}")
else:
    print(f"(Sum değil) {stage3}")

Stage 3 fix-point: 2 iter, 37 expand-step
    9× Pairing R-linearity
    6× Koszul expand
    6× d R-linearity
    3× L_X R-linearity in arg
    3× L_X ∘ d = d ∘ L_X (flow)
    3× Pairing-Lie Leibniz
    3× L_X R-linearity in X
    3× [L_X, L_Y] = L_{[X,Y]_VF}
    1× [π,π]_SN bivector formula

Toplam terim: 19  (1 adet SN handle, 18 residue)

  [ 0]    L_π♯(L_π♯(α)(β))(γ)
  [ 1]    (-L_π♯(L_π♯(α)(γ))(β))
  [ 2]    (-L_π♯(L_π♯(β)(α))(γ))
  [ 3]    L_π♯(L_π♯(β)(γ))(α)
  [ 4]    L_π♯(L_π♯(γ)(α))(β)
  [ 5]    (-L_π♯(L_π♯(γ)(β))(α))
  [ 6]    (-L_π♯(d(⟨π♯(α), β⟩))(γ))
  [ 7]    (-L_π♯(d(⟨π♯(β), γ⟩))(α))
  [ 8]    (-L_π♯(d(⟨π♯(γ), α⟩))(β))
  [ 9]  ★ [·,·]_SN(π, π)
  [10]    d(⟨L_π♯(α)(π♯(β)), γ⟩)
  [11]    d(⟨L_π♯(β)(π♯(γ)), α⟩)
  [12]    d(⟨L_π♯(γ)(π♯(α)), β⟩)
  [13]    d(⟨π♯(α), L_π♯(β)(γ)⟩)
  [14]    (-d(⟨π♯(α), d(⟨π♯(β), γ⟩)⟩))
  [15]    d(⟨π♯(β), L_π♯(γ)(α)⟩)
  [16]    (-d(⟨π♯(β), d(⟨π♯(γ), α⟩)⟩))
  [17]    d(⟨π♯(γ), L_π♯(α)(β)⟩)
  [18]    (-d(⟨π♯(γ), d(⟨π♯(α), β⟩)⟩))


In [8]:
from jacopy.brackets.schouten import SchoutenBracket

# SN bracket'i doğrudan π, π üzerinde açmaya çalış
sn_direct = SchoutenBracket().expand(pi, pi, registry=reg)
print(f"SchoutenBracket().expand(π, π, registry=reg)  →  {sn_direct}")
print(f"  type: {type(sn_direct).__name__}")
print()

# Stage 3 residue'undaki SN handle'ı bul
sn_handles = [c for c in stage3.children if isinstance(c, BracketApply)]
assert len(sn_handles) == 1
sn_handle = sn_handles[0]
print(f"Stage 3 residue'undaki SN handle  →  {sn_handle}")
print(f"  type: {type(sn_handle).__name__}")
print()

print(f"Eşit mi (structural)? {sn_direct == sn_handle}")
print()
print("→ Atomic π için SchoutenBracket base case'lerden hiçbirine düşmez (1-vector / function bekler)")
print("  ve wedge Leibniz tetiklenmez (π Product değil). Sonuç opaque BracketApply olarak döner —")
print("  yani Stage 2/3'teki recognizer-folding ile aynı token. Daha derin bir term-level açılım için")
print("  π'nin somut bir wedge olarak verilmesi (örn. e1∧e2) ya da koordinat seçimi gerekir.")

SchoutenBracket().expand(π, π, registry=reg)  →  [·,·]_SN(π, π)
  type: BracketApply

Stage 3 residue'undaki SN handle  →  [·,·]_SN(π, π)
  type: BracketApply

Eşit mi (structural)? True

→ Atomic π için SchoutenBracket base case'lerden hiçbirine düşmez (1-vector / function bekler)
  ve wedge Leibniz tetiklenmez (π Product değil). Sonuç opaque BracketApply olarak döner —
  yani Stage 2/3'teki recognizer-folding ile aynı token. Daha derin bir term-level açılım için
  π'nin somut bir wedge olarak verilmesi (örn. e1∧e2) ya da koordinat seçimi gerekir.


## 7. SN tarafının açılımı — neden residue burada duruyor?

`[π,π]_SN` bracket'ini doğrudan `SchoutenBracket.expand(π, π)` ile çağırdığımızda (yukarıdaki hücre): atomic π bir bivector olduğu için Schouten bracket *base case*'lerden hiçbirine düşmez (tüm base case'ler 1-vector / function operandı bekler) ve wedge Leibniz tetiklenmez (π wedge product değil) — sonuç **opaque `BracketApply([·,·]_SN, π, π)`** olarak döner.

Yani Stage 2/3'te engine'ın sıfırlamadığı 18 residue terimi, *SN-pairing'in evaluator-parçalarıdır* — bunların `0`'a düşmesi `[π,π]_SN = 0` (ya da pairing değerinin sıfır olması) **ek bir hipotez** olur, axiom değil.

### Residue'un anatomisi (Stage 3, 18 terim)

| Aile | Terim | Geometrik anlam |
|---|---|---|
| **Nested-VF L** (terimler 0-5) | `L_{L_π♯a(b)}(c)` | π♯ Sharp linearity'sinin nested 1-form'a uygulandığı blok; bunlar SN'in *anchor-content* parçası |
| **Hamiltonian-VF L** (terimler 6-8) | `L_{π♯(d⟨π♯a,b⟩)}(c)` | π♯ ile exact 1-form bileşimi → `X_{⟨π♯a,b⟩}` Hamiltonian VF; SN'in *Hamiltonian-content* parçası |
| **d-LieBracket-VF pairing** (terimler 10-12) | `d⟨L_π♯a(π♯b), c⟩ = d⟨[π♯a, π♯b]_VF, c⟩` | LieBracket-VF triple'ın d-pairing recognizer'ı (Faz 13.F adayı) |
| **d-Pairing-Lie residue** (terimler 13, 15, 17) | `d⟨π♯a, L_π♯b(c)⟩` | Pairing-Lie-Leibniz'in d-katmanlı kalıntısı |
| **d² nested** (terimler 14, 16, 18) | `d⟨π♯a, d⟨π♯b, c⟩⟩` | İki nested d sayesinde dış d²=0 *değil*, çünkü iç d ⟨π♯b,c⟩'ye uygulanmış (skaler) |

### Nasıl tam kapanırdı?

Engine residue'u `0`'a indirmek için ek recognizer axiom'lar gerekir — Faz 13.F kapsamı:

1. **`d⟨[π♯a, π♯b]_VF, c⟩` cyclic recognizer** → ikinci bir `d⟨[π,π]_SN, …⟩` token'ı.
2. **Hamiltonian-VF açılımı** (13.E paketinden form-level versiyonu): `L_{X_f}(g) = {f, g}_π` yorumu.
3. **Pairing identity cycler** (`L_X⟨α,Y⟩` ↔ `⟨L_X α, Y⟩ + ⟨α, L_X Y⟩` zincirleri arasında 0'lanan komutator pair'leri).

Tam kapanma `Σ ham 27 terim ≡ ½⟨[π,π]_SN, α∧β∧γ⟩` özdeşliğinin operator-level ispatına denk gelir — *engine seviyesinde Derived Bracket Theorem*. Bu defter **birinci recognizer'ı** açıkça yakalar (terimler 9'daki `★` SN handle); kalan recognizer aileleri Faz 13.F roadmap'ine ayrılmıştır.

## Sonuç

$$
\boxed{\;\sum_{\text{cyc}} L_{[\pi^\sharp a, \pi^\sharp b]_{VF}}(c) \;\stackrel{\text{Axiom 6}}{=}\; \tfrac12 [\cdot,\cdot]_{SN}(\pi, \pi) \;\text{(α∧β∧γ üzerinde değerlendirildiğinde)}.\;}
$$

* **Stage 1**: 27 ham terim — tam görünür.
* **Stage 2**: 6 terim → 1 SN handle; 22 residue (farklı evaluator aileleri).
* **Stage 3**: `[L,d]=0` + `d²=0` ile residue 18'e iner; 1 SN handle korunur.

2f-theo'da `prove_koszul_jacobi_reduction` aynı kapanmayı 1-adımda DerivedBracketTheorem'i çağırarak verir; 2f-deep buradaki 6+2+2 axiom'un **term-level** çalışmasıyla aynı gerçeği ispatlar — *engine seviyesinde Derived Bracket Theorem ispatı*.

## 2f-deep ile 2g-deep ilişkisi

2g-deep aynı sonucu **fonksiyon-level cyclic Poisson Jacobi sum**'ı 27-terim üzerinden ispatlar — 2f-deep'in function-ikizi. Faz 13.E `examples/2g-deep.ipynb` kapsamına ayrılmıştır.

Faz 13'ün 6+2 axiom paketi şu paylaşıma sahiptir:

* **Reused** (2g-deep tarafından): 13.A axiom 2 (`π^♯(df) → X_f`), 13.C tamamı (LieBracketVF + Op-commutator + Lie-Jacobi-VF).
* **Skipped**: 13.B (Pairing axiom'ları — fonksiyon zincirinde nested 1-form pairing yok).
* **Replaced**: 13.D form-level SN formülü → 13.E fonksiyon-level Hamiltonian morphism failure axiom'u.